In [11]:
from qiskit.circuit.library import CXGate, RXXGate, SwapGate, iSwapGate
import numpy as np
from gulps.invariants import (
    mono_coordinates_to_CAN,
    mono_coordinates_to_makhlin,
    positive_canonical_to_monodromy_coordinate,
    recover_local_equivalence,
)
from gulps.lm_numerics import _SegmentNumericSynthesis

In [ ]:
# Example usage
# Define a gate list and invariant list for testing

gate_list = [CXGate(), CXGate(), CXGate()]
invariant_list = np.pi / 2 * np.array([(0.5, 0.5, 0.0), (0.5, 0.5, 0), (0.5, 0.5, 0.5)])
invariant_list = [
    positive_canonical_to_monodromy_coordinate(*c) for c in invariant_list
]

# Synthesize segments
segment_solutions = _SegmentNumericSynthesis._synthesize_segments(
    gate_list, invariant_list
)
print("Segment solutions:", segment_solutions)

Segment solutions: [array([-0.92806971, -6.14288956,  0.93915506,  3.05371084,  2.4953907 ,
        2.0420265 ]), array([-4.17395110e-03,  4.71238450e+00, -4.17408115e-03,  2.40924366e+00,
        2.45528907e+00,  2.40924366e+00])]


In [13]:
import jax.numpy as jnp
from gulps.lm_numerics import _rv

jnp.kron(
    _rv(jnp.array([0.70127154, 4.58352926, -0.70072199])),
    _rv(jnp.array([3.05288158, 2.49471306, 2.04311779])),
)

Array([[ 0.46186141+0.19150739j,  0.37125429+0.3349049j ,
         0.38352335+0.3208096j ,  0.2541936 +0.4305747j ],
       [-0.25423375+0.43053018j,  0.38354631-0.32075422j,
        -0.37130591+0.33487443j,  0.46189987-0.19146141j],
       [-0.46189987-0.19146141j, -0.37130591-0.33487443j,
         0.38354631+0.32075422j,  0.25423375+0.43053018j],
       [ 0.2541936 -0.4305747j , -0.38352335+0.3208096j ,
        -0.37125429+0.3349049j ,  0.46186141-0.19150739j]],      dtype=complex128)

In [14]:
ret = _SegmentNumericSynthesis._stitch_segments(
    gate_list, invariant_list, segment_solutions
)
from weylchamber import c1c2c3
from qiskit.quantum_info import Operator

print(c1c2c3(Operator(ret).data))
ret.draw()

(0.5, 0.5, 0.49999999)


global phase: π
     ┌─────────┐┌─────────┐┌────────┐  ┌─────────────────────────┐   ┌────────┐»
q_0: ┤ Unitary ├┤ Unitary ├┤0       ├──┤ Rv(3.0537,2.4954,2.042) ├───┤0       ├»
     ├─────────┤├─────────┤│  Iswap │┌─┴─────────────────────────┴──┐│  Iswap │»
q_1: ┤ Unitary ├┤ Unitary ├┤1       ├┤ Rv(-0.92807,-6.1429,0.93916) ├┤1       ├»
     └─────────┘└─────────┘└────────┘└──────────────────────────────┘└────────┘»
«     ┌─────────┐    ┌──────────────────────────┐   ┌────────┐┌─────────┐
«q_0: ┤ Unitary ├────┤ Rv(2.4092,2.4553,2.4092) ├───┤0       ├┤ Unitary ├
«     ├─────────┤┌───┴──────────────────────────┴──┐│  Iswap │├─────────┤
«q_1: ┤ Unitary ├┤ Rv(-0.004174,4.7124,-0.0041741) ├┤1       ├┤ Unitary ├
«     └─────────┘└─────────────────────────────────┘└────────┘└─────────┘

In [15]:
from qutip import Qobj
from qiskit.quantum_info import Operator, average_gate_fidelity

lm_synthesizer = _SegmentNumericSynthesis()
decomp = lm_synthesizer(gate_list, invariant_list, target_unitary=SwapGate())
decomp_u = Operator(decomp)
decomp.draw()
print(average_gate_fidelity(SwapGate(), decomp_u))
Qobj(decomp_u.data)

1.0000000000000007


Quantum object: dims=[[4], [4]], shape=(4, 4), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 1.00000000e+00-4.53933835e-09j  7.15222326e-12+2.63788991e-13j
   7.15244753e-12+2.63629833e-13j -1.01909042e-09+1.37705757e-08j]
 [-7.15194999e-12+2.64955040e-13j  1.39219700e-17-1.38082336e-08j
   1.00000000e+00+4.53933829e-09j -7.15266735e-12-2.64510636e-13j]
 [-7.15266735e-12+2.64899214e-13j  1.00000000e+00+4.53933846e-09j
   9.54316225e-17-1.38082336e-08j -7.15232338e-12-2.64993222e-13j]
 [ 1.01909013e-09+1.37705757e-08j  7.15296910e-12-2.63795087e-13j
   7.15225101e-12-2.63900013e-13j  1.00000000e+00-4.53933871e-09j]]